# 📝 [제 1편] 머신러닝을 위한 미분 기초

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「머신러닝/딥러닝을 위한 수치미분 (I) - 미분 • 편미분 • 체인룰」 (ML_LEC_09)

---

## 🎯 학습 목표
1. 미분의 의미를 "입력에 대한 출력의 민감도" 관점에서 설명할 수 있다.
2. 다항함수, 지수함수, 로그함수의 미분 공식을 알고 계산할 수 있다.
3. 편미분과 연쇄법칙(chain rule)의 뜻을 이해하고 계산할 수 있다.

---

## 🧭 지난 시간 복습
> **오늘의 질문** 🤔 인공지능은 어떻게 "똑똑해질까"?  
> → 틀린 정도(오차)를 줄이는 방향으로 자기 안의 숫자(가중치)를 조금씩 고칩니다.  
> 　 그런데 **어느 방향으로, 얼마나** 고쳐야 할까? 그것을 알려 주는 도구가 바로 **미분** 입니다.

[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 2편 수치미분 - 파이썬으로 미분하기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/02_numerical_derivative.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 미분이란? - 입력에 대한 민감도

> **f′(x) = lim<sub>Δx→0</sub> {f(x + Δx) − f(x)} / Δx**

미분은 한마디로

> **"입력 x 를 지금 값에서 아주 조금 바꾸면, 출력 f(x) 는 얼마나 변하는가?"**
> = 함수 f(x) 가 입력의 미세한 변화에 **얼마나 민감하게 반응하는가?**

### [이해해 보기] f(x) = x² 이면 f′(x) = 2x
* f(3) = 9 → 입력 x = 3 일 때 출력은 9
* f′(3) = 6 → x = 3 에서 x 를 아주 조금 변화시키면, f(x) 는 그 변화량의 ( 　 ) 배만큼 변한다.

> **[확인 1]** x 를 3 에서 3.01 로 0.01 만큼 늘리면 f(x) 는 약 6 × 0.01 = ( 　　 ) 만큼 늘어난다.
> 실제로 계산하면 f(3.01) − f(3) = 9.0601 − 9 = ( 　　　 ) 으로 거의 같다!

In [ ]:
f = lambda x : x**2
for dx in [0.1, 0.01, 0.001]:
    real = f(3 + dx) - f(3)
    print(f'Δx = {dx:<6} :  실제 변화량 f(3+Δx) − f(3) = {real:.6f},   f′(3) × Δx = {6*dx:.6f},   비율 = {real/dx:.4f}')

> 💡 Δx 가 작아질수록 (변화량 ÷ Δx) 가 6 에 가까워집니다. 이 값이 바로 **미분계수 f′(3) = 6** 입니다.

## 🎬 애니메이션 1. 곳곳의 민감도 = 도함수

점이 f(x) = x² 위를 움직이면서 그 점의 **접선의 기울기 (민감도)** 를 오른쪽 그래프에 찍습니다. 찍힌 점들을 이으면 **도함수 f′(x) = 2x** 가 됩니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (민감도와 도함수)
f  = lambda x : x**2
df = lambda x : 2*x
xs_path = np.linspace(-2.5, 2.5, 51)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8))
X = np.linspace(-3, 3, 200)
def draw(i):
    x0 = xs_path[min(i, len(xs_path) - 1)]
    axL.clear(); axR.clear()
    axL.plot(X, f(X), color='royalblue', lw=2, label='f(x) = x²')
    tl = np.linspace(x0 - 1.2, x0 + 1.2, 10)
    axL.plot(tl, f(x0) + df(x0)*(tl - x0), color='crimson', lw=2.5, label=f'접선 기울기 = {df(x0):.1f}')
    axL.plot([x0], [f(x0)], 'o', color='crimson', ms=10, mec='k', zorder=5)
    axL.set_xlim(-3, 3); axL.set_ylim(-2, 9); axL.grid(alpha=0.3); axL.legend(loc='upper center')
    msg = '오른쪽으로 갈수록 f 가 줄어든다' if df(x0) < -0.05 else ('오른쪽으로 갈수록 f 가 늘어난다' if df(x0) > 0.05 else '여기서는 변화가 없다 (최솟값)')
    axL.set_title(f'x = {x0:.1f} 에서의 민감도 → {msg}', fontsize=12)
    done = xs_path[:min(i, len(xs_path) - 1) + 1]
    axR.plot(X, df(X), color='lightgray', lw=1, ls='--')
    axR.plot(done, df(done), 'o-', color='crimson', ms=4)
    axR.axhline(0, color='k', lw=0.6)
    axR.set_xlim(-3, 3); axR.set_ylim(-6, 6); axR.grid(alpha=0.3)
    axR.set_title("기울기를 모아 그린 그래프 = 도함수 f′(x) = 2x", fontsize=12)
anim = animation.FuncAnimation(fig, draw, frames=len(xs_path) + 8, interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 기울기가 음수인 곳에서 x 를 늘리면 f(x) 는 ( 늘어난다 / 줄어든다 ). 기울기가 0 인 곳 x = ( 　 ) 에서 f(x) 는 가장 ( 크다 / 작다 ).

* **머신러닝에서 미분을 왜 할까?**
  인공지능은 오차를 줄여야 합니다. 오차를 나타내는 함수의 **기울기(미분값)** 를 알면 "가중치를 늘려야 오차가 줄어드는지, 줄여야 하는지" 를 알 수 있습니다. (→ 제 6편 경사하강법)

## 📖 개념 다지기 2. 머신러닝에서 자주 쓰는 미분 공식

| 함수 | 도함수 |
|:-|:-|
| f(x) = c (상수) | f′(x) = 0 |
| f(x) = ax<sup>n</sup> | f′(x) = n·a·x<sup>n−1</sup> |
| f(x) = e<sup>x</sup> | f′(x) = e<sup>x</sup> (모양이 그대로!) |
| f(x) = e<sup>−x</sup> | f′(x) = −e<sup>−x</sup> |
| f(x) = ln x | f′(x) = 1/x |

> **[연습 1]** 다음 함수를 미분하세요.
> ① f(x) = 3x² + e<sup>x</sup> + 7　→　f′(x) = ( 　　　　　　 )
> ② f(x) = ln x + 1/x　→　f′(x) = ( 　　　　　　 )　(힌트: 1/x = x<sup>−1</sup>)
> ③ f(x) = 2x³ − 5x　→　f′(x) = ( 　　　　　　 )

아래 셀은 파이썬의 기호 계산 도구 **sympy** 로 답을 확인합니다. 손으로 먼저 풀고 실행하세요!

In [ ]:
import sympy as sp
x = sp.symbols('x')
for expr in [3*x**2 + sp.exp(x) + 7, sp.log(x) + 1/x, 2*x**3 - 5*x]:
    print(f'f(x) = {expr}   →   f′(x) = {sp.diff(expr, x)}')

## 📖 개념 다지기 3. 변수가 여러 개라면? 편미분

체중은 '야식' 과 '운동량' 둘 다의 영향을 받습니다.  →  체중 = f(야식, 운동량)

**편미분** : 여러 변수 중 **관심 있는 변수 1개만** 진짜 변수로 보고, **나머지는 상수로 고정** 하고 미분하는 것

| 기호 | 뜻 |
|:-:|:-|
| ∂체중 / ∂야식 | 운동량은 그대로 두고, 야식만 조금 바꿀 때 체중이 얼마나 변하나? |
| ∂체중 / ∂운동 | 야식은 그대로 두고, 운동량만 조금 바꿀 때 체중이 얼마나 변하나? |

> **[연습 2]** f(x, y) = 2x + 3xy + y³ 일 때
> ① ∂f/∂x (y 는 상수 취급) = ( 　　　　 )
> ② ∂f/∂y (x 는 상수 취급) = ( 　　　　 )
> ③ 점 (1, 2) 에서 ∂f/∂x = ( 　 ),  ∂f/∂y = ( 　 )

In [ ]:
x, y = sp.symbols('x y')
F = 2*x + 3*x*y + y**3
fx, fy = sp.diff(F, x), sp.diff(F, y)
print('∂f/∂x =', fx, '  → 점 (1, 2) 에서', fx.subs({x: 1, y: 2}))
print('∂f/∂y =', fy, '  → 점 (1, 2) 에서', fy.subs({x: 1, y: 2}))

## 🎬 애니메이션 2. 편미분 = 곡면을 한 방향으로 자른 단면의 기울기

곡면 f(x, y) = x² + y² 을 **y = 1 평면으로 자르면** 단면은 곡선 f(x, 1) = x² + 1 이 됩니다.
이 단면 곡선의 x = 1 에서의 접선 기울기가 바로 **∂f/∂x = 2x = 2** 입니다. (곡면이 천천히 돌아갑니다)

In [ ]:
#@title 🎬 애니메이션 2 실행 (편미분의 기하학적 의미)
Xg, Yg = np.meshgrid(np.linspace(-2, 2, 40), np.linspace(-2, 2, 40))
Zg = Xg**2 + Yg**2
xl = np.linspace(-2, 2, 60)
fig = plt.figure(figsize=(13, 5.2))
ax3 = fig.add_subplot(121, projection='3d'); ax2 = fig.add_subplot(122)
ax2.plot(xl, xl**2 + 1, color='darkorange', lw=2.5, label='단면 f(x, 1) = x² + 1')
tl = np.linspace(-0.2, 2, 10)
ax2.plot(tl, 2 + 2*(tl - 1), 'r--', lw=2, label='접선 (기울기 ∂f/∂x = 2)')
ax2.plot(1, 2, 'ro', ms=9); ax2.grid(alpha=0.3); ax2.legend(loc='upper center')
ax2.set_xlabel('x'); ax2.set_title('y = 1 로 자른 단면 (위에서 보면 2차원 곡선)', fontsize=12)
def draw(i):
    ax3.clear()
    ax3.plot_surface(Xg, Yg, Zg, cmap='viridis', alpha=0.45, linewidth=0)
    Xp, Zp = np.meshgrid(np.linspace(-2, 2, 2), np.linspace(0, 8, 2))
    ax3.plot_surface(Xp, np.ones_like(Xp), Zp, color='hotpink', alpha=0.25)
    ax3.plot(xl, np.ones_like(xl), xl**2 + 1, color='darkorange', lw=3)
    ax3.plot(tl, np.ones_like(tl), 2 + 2*(tl - 1), 'r--', lw=2)
    ax3.scatter([1], [1], [2], color='red', s=50)
    ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('f'); ax3.set_zlim(0, 8)
    ax3.set_title('f(x, y) = x² + y² 와 y = 1 평면', fontsize=12)
    ax3.view_init(elev=25, azim=-70 + i*8)
anim = animation.FuncAnimation(fig, draw, frames=36, interval=150)
plt.close(fig)
show_anim(anim)

> **[관찰 2]** 같은 점 (1, 1) 에서 **x = 1 평면** 으로 자르면 단면은 f(1, y) = ( 　　　 ) 이고, 그 기울기 ∂f/∂y = ( 　 ) 이다.

## 📖 개념 다지기 4. 양파 껍질 까듯이 - 연쇄법칙 (chain rule)

함수 안에 함수가 들어 있는 **합성함수** 는 **겉 함수의 미분 × 속 함수의 미분** 으로 구합니다.

> **df/dx = (df/dt) × (dt/dx)**　　(마치 분수의 약분처럼 dt 가 사라진다고 생각하면 쉽습니다)

### [예 1] f(x) = e<sup>3x²</sup>
1. 치환: t = 3x² 으로 놓으면 f = e<sup>t</sup>
2. 각각 미분:  df/dt = ( 　　 ),  dt/dx = ( 　　 )
3. 곱하기:  df/dx = e<sup>t</sup> × 6x = ( 　　　　 )

### [예 2] f(x) = e<sup>−x</sup>
t = −x 로 놓으면  df/dx = e<sup>t</sup> × ( 　 ) = −e<sup>−x</sup>

> 💡 딥러닝의 신경망은 함수가 **겹겹이 쌓인 거대한 합성함수** 입니다. 그래서 연쇄법칙이 딥러닝 학습(오차역전파, 제 12 ~ 13편)의 핵심입니다!

In [ ]:
x = sp.symbols('x')
for expr in [sp.exp(3*x**2), sp.exp(-x), sp.log(x**2 + 1)]:
    print(f'f(x) = {expr}   →   f′(x) = {sp.diff(expr, x)}')

## 🎬 애니메이션 3. 변화가 전달되는 연쇄법칙

f(x) = e<sup>3x²</sup> 에서 x = 0.5 근처를 생각합니다. x 가 조금 흔들리면 → t = 3x² 가 흔들리고 → f = e<sup>t</sup> 가 흔들립니다.
각 단계에서 흔들림이 **몇 배로 커지는지** (dt/dx, df/dt) 곱하면 전체 배율 df/dx 가 됩니다.

In [ ]:
#@title 🎬 애니메이션 3 실행 (연쇄법칙)
x0 = 0.5; t0 = 3*x0**2; f0 = np.exp(t0)
dt_dx = 6*x0; df_dt = np.exp(t0); df_dx = dt_dx * df_dt
amp = 0.05
ks = np.sin(np.linspace(0, 4*np.pi, 60))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
fig.subplots_adjust(top=0.72)
def draw(i):
    dx = amp*ks[i]; dt = 3*(x0 + dx)**2 - t0; dfv = np.exp(t0 + dt) - f0
    for ax in axes: ax.clear(); ax.set_ylim(-0.75, 0.75); ax.axhline(0, color='k', lw=0.6); ax.set_xticks([])
    axes[0].bar([0], [dx], color='royalblue', width=0.5); axes[0].set_title(f'① x 의 변화 Δx = {dx:+.3f}', fontsize=12)
    axes[1].bar([0], [dt], color='darkorange', width=0.5); axes[1].set_title(f'② t = 3x² 의 변화 Δt = {dt:+.3f}\n(약 dt/dx = {dt_dx:.1f} 배)', fontsize=12)
    axes[2].bar([0], [dfv], color='crimson', width=0.5); axes[2].set_title(f'③ f = e^t 의 변화 Δf = {dfv:+.3f}\n(Δt 의 약 df/dt = {df_dt:.2f} 배)', fontsize=12)
    fig.suptitle(f'전체 배율 df/dx = (dt/dx) × (df/dt) = {dt_dx:.1f} × {df_dt:.2f} = {df_dx:.2f}', fontsize=14, y=0.98)
anim = animation.FuncAnimation(fig, draw, frames=len(ks), interval=100)
plt.close(fig)
show_anim(anim)

> **[관찰 3]** Δx = 0.05 일 때 Δf 는 약 ( 　　 ) 이다. 이것은 Δx 의 약 ( 　　 ) 배이며, 공식 6x·e<sup>3x²</sup> 에 x = 0.5 를 넣은 값 ( 　　 ) 과 같다.

## ✏️ 연습 문제

> **[문제 1]** f(x) = 4x³ − 2x + ln x 의 도함수와 f′(1) 을 구하세요.  f′(x) = ( 　　　　　 ),  f′(1) = ( 　 )

> **[문제 2]** f(x, y) = x²y + e<sup>y</sup> 의 ∂f/∂x, ∂f/∂y 를 구하세요.  ( 　　 ),  ( 　　 )

> **[문제 3]** f(x) = e<sup>−2x</sup> 을 연쇄법칙으로 미분하세요. t = ( 　　 ) 로 놓으면 f′(x) = ( 　　　 )

아래 셀의 `______` 에 식을 넣어 sympy 로 확인하세요. (e<sup>x</sup> 는 `sp.exp(x)`, ln x 는 `sp.log(x)`)

In [ ]:
x, y = sp.symbols('x y')
f1 = ______
print('문제 1 :', sp.diff(f1, x), ',  f′(1) =', sp.diff(f1, x).subs(x, 1))
f2 = ______
print('문제 2 :', sp.diff(f2, x), ',', sp.diff(f2, y))
f3 = ______
print('문제 3 :', sp.diff(f3, x))

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print("문제 1 : f′(x) = 12x² − 2 + 1/x,  f′(1) = 12 − 2 + 1 = 11")
print("문제 2 : ∂f/∂x = 2xy,  ∂f/∂y = x² + e^y")
print("문제 3 : t = −2x,  f′(x) = e^t × (−2) = −2e^(−2x)")

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 미분 | 입력을 아주 조금 바꿀 때 출력이 얼마나 변하는지 (민감도) |
| 공식 | (ax<sup>n</sup>)′ = nax<sup>n−1</sup>, (e<sup>x</sup>)′ = e<sup>x</sup>, (ln x)′ = 1/x |
| 편미분 | 한 변수만 움직이고 나머지는 상수로 고정 |
| 연쇄법칙 | df/dx = (df/dt) × (dt/dx) |

### 🏆 도전 과제
컴퓨터는 "x² 을 미분하면 2x" 라는 공식을 모릅니다. 그렇다면 컴퓨터로 f′(3) 을 구하려면 어떻게 하면 될까요? (힌트: 확인 1 의 표)
→ 다음 시간: **수치미분 - 파이썬으로 미분하기**

(                                                                                                                            )

---

[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 2편 수치미분 - 파이썬으로 미분하기 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/02_numerical_derivative.ipynb)